In [35]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier 
from sklearn.metrics import accuracy_score , classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
import mlflow
from pathlib import Path


import matplotlib.pyplot as plt
import seaborn as sns

In [36]:
TRAIN_PATH = "./../data/processed/train.csv"
VALIDATION_PATH = "./../data/processed/validation.csv"
TEST_PATH = "./../data/processed/test.csv"

In [37]:
train_df= pd.read_csv(TRAIN_PATH)
validation_df= pd.read_csv(VALIDATION_PATH)
test_df= pd.read_csv(TEST_PATH)

X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]

X_validation = validation_df.drop(columns=["label"])
y_validation = validation_df["label"]

X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]

del train_df, validation_df, test_df

In [38]:
#keep turnid and convid for later use in error analysis and model explainability
X_train_ids = X_train[["turn_id", "conv_id"]]
X_validation_ids = X_validation[["turn_id", "conv_id"]]
X_test_ids = X_test[["turn_id", "conv_id"]]

X_train = X_train.drop(columns=["turn_id", "conv_id"])
X_validation = X_validation.drop(columns=["turn_id", "conv_id"])
X_test = X_test.drop(columns=["turn_id", "conv_id"])

In [39]:
print("Training set shape:", X_train.shape)
print("Validation set shape:", X_validation.shape)
print("Test set shape:", X_test.shape)

Training set shape: (2636, 1554)
Validation set shape: (293, 1554)
Test set shape: (326, 1554)


In [40]:
def save_confusion_matrix(cm, filename,title):
    plt.figure()
    sns.heatmap(cm, annot=True, fmt="d") 
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.savefig(filename)

    plt.close()

In [41]:
def evaluate_model(y_pred, y_true):
    acc = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    return acc, report, cm

In [42]:
from evaluate import MultiTurnJailbreakEvaluator
import pandas as pd

evaluator = MultiTurnJailbreakEvaluator()

def group_by_conv(X, y_true, y_pred):
 
    df = X.copy()
    df["_label"] = y_true
    df["_pred"]  = y_pred

    # sort so turns are in order within each conversation
    df = df.sort_values(["conv_id"]).reset_index(drop=True)

    conv_ids       = []
    y_true_grouped = []
    y_pred_grouped = []

    for cid, group in df.groupby("conv_id", sort=False):
        conv_ids.append(cid)
        y_true_grouped.append(group["_label"].tolist())
        y_pred_grouped.append(group["_pred"].tolist())

    return conv_ids, y_true_grouped, y_pred_grouped


In [43]:
from sklearn.neighbors import LocalOutlierFactor

lof = LocalOutlierFactor(
    n_neighbors=min(10, len(X_train)-1),
    contamination=.2,
    metric="cosine"
)

y_pred_train = lof.fit_predict(X_train, y_train)
y_pred_train = [1 if p == -1 else 0 for p in y_pred_train]
acc_train, report_train, cm_train = evaluate_model(y_pred_train, y_train)



In [44]:
print ("LOF Train Accuracy:", acc_train)
print ("LOF Train Classification Report:\n", report_train)

LOF Train Accuracy: 0.6820940819423369
LOF Train Classification Report:
               precision    recall  f1-score   support

           0       0.76      0.83      0.79      1923
           1       0.38      0.28      0.32       713

    accuracy                           0.68      2636
   macro avg       0.57      0.56      0.56      2636
weighted avg       0.66      0.68      0.67      2636



In [45]:

params = {
    'penalty': 'l2',         
    'C': 1.0,                
    'solver': 'lbfgs',       
    'max_iter': 10000,        
    'random_state': 42
}

logreg = LogisticRegression(**params)
logreg.fit(X_train, y_train)

y_train_pred = logreg.predict(X_train)
y_validation_pred = logreg.predict(X_validation)

train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)


test_acc, test_report, test_cm = evaluate_model(logreg.predict(X_test), y_test)


Training Accuracy: 0.7534142640364189
Validation Accuracy: 0.7167235494880546
Training Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.95      0.85      1923
           1       0.63      0.22      0.32       713

    accuracy                           0.75      2636
   macro avg       0.70      0.58      0.59      2636
weighted avg       0.73      0.75      0.71      2636

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.92      0.82       209
           1       0.51      0.21      0.30        84

    accuracy                           0.72       293
   macro avg       0.63      0.57      0.56       293
weighted avg       0.68      0.72      0.67       293



d:\Apps\anaconda\envs\langgraph-env\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 10000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=10000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [46]:
with open("../reports/evaluation_report.txt", "a") as f:
    # write data of the model 
    
    f.write("==============================\n" \
    "Model:logistic_regression \n" \
    "==============================\n" \
    "param_grid: {'C': 1.0, 'random_state': 42}\n" )
    # f.write("cv_best_score: " + str(grid_search.best_score_) + "\n" )
    f.write("train_acc: " + str(train_acc) + "\n" )
    f.write("val_acc: " + str(val_acc) + "\n" )
    f.write("train_report: " + str(train_report) + "\n" )
    f.write("val_report: " + str(val_report) + "\n" )
    f.write("test_acc: " + str(test_acc) + "\n" )
    f.write("test_report: " + str(test_report) + "\n" )
    


In [47]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)
print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)


Training Accuracy: 0.7534142640364189
Validation Accuracy: 0.7167235494880546
Training Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.95      0.85      1923
           1       0.63      0.22      0.32       713

    accuracy                           0.75      2636
   macro avg       0.70      0.58      0.59      2636
weighted avg       0.73      0.75      0.71      2636

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.92      0.82       209
           1       0.51      0.21      0.30        84

    accuracy                           0.72       293
   macro avg       0.63      0.57      0.56       293
weighted avg       0.68      0.72      0.67       293

Test Accuracy: 0.7116564417177914
Test Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.91      0.81       226
           1       0.56     

In [48]:
params = {
    'max_depth': 5,           
    'min_samples_split': 5,    
    'min_samples_leaf': 10,     
    'max_features': None,      
    'random_state': 42
}

dtc = DecisionTreeClassifier(**params)
dtc.fit(X_train, y_train)

y_train_pred = dtc.predict(X_train)
y_validation_pred = dtc.predict(X_validation)

train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)


mlflow.log_params(params)
mlflow.log_metrics({
    'train_acc': train_acc,
    'val_acc': val_acc
})

test_acc, test_report, test_cm = evaluate_model(dtc.predict(X_test), y_test)

In [49]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)
print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)


Training Accuracy: 0.8679817905918058
Validation Accuracy: 0.7747440273037542
Training Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.89      0.91      1923
           1       0.73      0.81      0.77       713

    accuracy                           0.87      2636
   macro avg       0.83      0.85      0.84      2636
weighted avg       0.87      0.87      0.87      2636

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.84      0.84       209
           1       0.61      0.61      0.61        84

    accuracy                           0.77       293
   macro avg       0.72      0.72      0.72       293
weighted avg       0.77      0.77      0.77       293

Test Accuracy: 0.8098159509202454
Test Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.84      0.86       226
           1       0.67     

In [50]:
with open("evaluation_report.txt", "a") as f:
    # write data of the model 
    
    f.write("==============================\n" \
    "Model:decision_tree \n" \
    "==============================\n" \
    "param_grid: {'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 2 , 'random_state': 42 ,''}\n" )

    f.write("train_acc: " + str(train_acc) + "\n" )
    f.write("val_acc: " + str(val_acc) + "\n" )
    f.write("train_report: " + str(train_report) + "\n" )
    f.write("val_report: " + str(val_report) + "\n" )
    f.write("test_report: " + str(test_report) + "\n" )


In [51]:
params = {
'n_estimators': 200,          
'max_depth': 5,             
'min_samples_split': 5,      
'min_samples_leaf': 5,        
'max_features': 'sqrt',       
'bootstrap': True,
'random_state': 42,
'n_jobs': -1                 
}

rfc = RandomForestClassifier(**params)
rfc.fit(X_train, y_train)
y_train_pred = rfc.predict(X_train)
y_validation_pred = rfc.predict(X_validation)

train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)


test_acc, test_report, test_cm = evaluate_model(rfc.predict(X_test), y_test)

In [52]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)  
print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)  

Training Accuracy: 0.8399089529590288
Validation Accuracy: 0.7235494880546075
Training Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.99      0.90      1923
           1       0.96      0.42      0.59       713

    accuracy                           0.84      2636
   macro avg       0.89      0.71      0.75      2636
weighted avg       0.86      0.84      0.82      2636

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.96      0.83       209
           1       0.57      0.14      0.23        84

    accuracy                           0.72       293
   macro avg       0.65      0.55      0.53       293
weighted avg       0.69      0.72      0.66       293

Test Accuracy: 0.7300613496932515
Test Classification Report:
               precision    recall  f1-score   support

           0       0.73      0.96      0.83       226
           1       0.71     

In [53]:
with open("evaluation_report.txt", "a") as f:
    # write data of the model 
    
    f.write("==============================\n" \
    "Model:random_forest_tuned_v1 \n" \
    "==============================\n" \
    "param_grid: {'n_estimators': 300,'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 2,  'max_features': 'sqrt','bootstrap': True,'random_state': 42,'n_jobs': -1}'\n" )
    f.write("train_acc: " + str(train_acc) + "\n" )
    f.write("val_acc: " + str(val_acc) + "\n" )
    f.write("train_report: " + str(train_report) + "\n" )
    f.write("val_report: " + str(val_report) + "\n" )
    f.write("test_report: " + str(test_report) + "\n" )



In [54]:
params = {
    'n_estimators': 300,        
    'max_depth': 3,             
    'learning_rate': 0.01,       
    'random_state': 42,
    'n_jobs': -1,
}

xgb = XGBClassifier(**params)
xgb.fit(X_train, y_train)

# Predictions
y_train_pred = xgb.predict(X_train)
y_validation_pred = xgb.predict(X_validation)

# Evaluation
train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)
test_acc, test_report, test_cm = evaluate_model(xgb.predict(X_test), y_test)

In [55]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)
print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)

Training Accuracy: 0.9070561456752656
Validation Accuracy: 0.8088737201365188
Training Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.96      0.94      1923
           1       0.88      0.76      0.82       713

    accuracy                           0.91      2636
   macro avg       0.90      0.86      0.88      2636
weighted avg       0.91      0.91      0.90      2636

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.91      0.87       209
           1       0.71      0.56      0.63        84

    accuracy                           0.81       293
   macro avg       0.77      0.73      0.75       293
weighted avg       0.80      0.81      0.80       293

Test Accuracy: 0.8190184049079755
Test Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.92      0.88       226
           1       0.77     

In [56]:
with open("evaluation_report.txt", "a") as f:
    # write data of the model 
    
    f.write("==============================\n" \
    "Model:xgboost \n" \
    "==============================\n" \
    "param_grid: " + str(params) + "\n" )
    f.write("train_acc: " + str(train_acc) + "\n" )
    f.write("val_acc: " + str(val_acc) + "\n" )
    f.write("train_report: " + str(train_report) + "\n" )
    f.write("val_report: " + str(val_report) + "\n" )
    f.write("test_report: " + str(test_report) + "\n" )
    

In [57]:
params = {
    'C': 1.0,                
    'kernel': 'rbf',        
    'random_state': 42
}

svc = SVC(**params)
svc.fit(X_train, y_train)

y_train_pred = svc.predict(X_train)
y_validation_pred = svc.predict(X_validation)

train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

test_acc, test_report, test_cm = evaluate_model(svc.predict(X_test), y_test)



In [58]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)
print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)

Training Accuracy: 0.738619119878604
Validation Accuracy: 0.7167235494880546
Training Classification Report:
               precision    recall  f1-score   support

           0       0.74      1.00      0.85      1923
           1       0.79      0.05      0.09       713

    accuracy                           0.74      2636
   macro avg       0.76      0.52      0.47      2636
weighted avg       0.75      0.74      0.64      2636

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.72      0.99      0.83       209
           1       0.60      0.04      0.07        84

    accuracy                           0.72       293
   macro avg       0.66      0.51      0.45       293
weighted avg       0.68      0.72      0.61       293

Test Accuracy: 0.7024539877300614
Test Classification Report:
               precision    recall  f1-score   support

           0       0.70      1.00      0.82       226
           1       1.00      

In [59]:
with open("evaluation_report.txt", "a") as f:
   
    f.write("==============================\n" \
    "Model: Support Vector Classifier\n" \
    "==============================\n" \
    "param_grid: " + str(params) + "\n" )
    f.write("train_acc: " + str(train_acc) + "\n" )
    f.write("val_acc: " + str(val_acc) + "\n" )
    f.write("train_report: " + str(train_report) + "\n" )
    f.write("val_report: " + str(val_report) + "\n" )
    f.write("test_report: " + str(test_report) + "\n" )
